# RSOD reproduction on Colab (UATD)

Reproduces the UATD rows of **RSOD** (Li et al., AAAI 2026): labels-only Faster R-CNN vs. RSOD at 1/5/10 % labels.

**Before you start:** `Runtime → Change runtime type → GPU`. Then run the cells from top to bottom.

* Everything that must survive a disconnect lives on Google Drive (`DRIVE_ROOT`): the converted dataset, checkpoints and results.
* If Colab disconnects, **reconnect and run all cells again**: training resumes from the last checkpoint, and finished runs are skipped.
* Steps 1–4 take a few minutes each session. Step 3 (data) is slow only the first time.

In [ ]:
#@title 0. Settings (edit these)
GITHUB_REPO   = "YOUR_GITHUB_USERNAME/sonar-habitat-repro"   #@param {type:"string"}
PCTS          = "5"      #@param {type:"string"}   # e.g. "1 5 10"
SEEDS         = "1"      #@param {type:"string"}   # e.g. "1 2 3"
METHODS       = "sup rsod"  #@param {type:"string"}
ITERS         = ""       #@param {type:"string"}   # "" = paper schedule (180k). e.g. "2000" for a quick test
CKPT_INTERVAL = "2000"   #@param {type:"string"}   # save often so a disconnect loses little
DRIVE_ROOT    = "/content/drive/MyDrive/sonar-habitat"  #@param {type:"string"}
RSOD_COMMIT   = "4e41dfb"  # RSOD commit this kit was written against

import os
os.environ.update(GITHUB_REPO=GITHUB_REPO, PCTS=PCTS, SEEDS=SEEDS, METHODS=METHODS, ITERS=ITERS,
                  CKPT_INTERVAL=CKPT_INTERVAL, DRIVE_ROOT=DRIVE_ROOT, RSOD_COMMIT=RSOD_COMMIT)

In [ ]:
#@title 1. Mount Google Drive and check the GPU
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_ROOT, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title 2. Get the code (your repo + RSOD)
# Private repo? Add a GitHub token as a Colab secret named GITHUB_TOKEN (key icon on the left).
token = ""
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ""
except Exception:
    pass
url = f"https://{token+'@' if token else ''}github.com/{GITHUB_REPO}.git"
if os.path.isdir('/content/kit/.git'):
    !git -C /content/kit pull -q
else:
    !git clone -q {url} /content/kit
if not os.path.isdir('/content/RSOD/.git'):
    !git clone -q https://github.com/LICZ9/RSOD.git /content/RSOD
!git -C /content/RSOD checkout -q {RSOD_COMMIT} && echo "RSOD at $(git -C /content/RSOD rev-parse --short HEAD)"
!ls /content/kit

In [ ]:
#@title 3. Python 3.10 environment with RSOD's versions (torch 2.0.0 + CUDA 11.8, mmcv 2.1.0, mmengine 0.10.6)
# The commands live in the repo (colab/setup_env.sh), so cell 2's git pull brings any fixes.
!bash /content/kit/colab/setup_env.sh

In [ ]:
#@title 4. Dataset: download UATD once, convert to lossless PNG + COCO on Drive, copy to local disk
# The logic lives in the repo (colab/prepare_uatd.py), so cell 2's git pull brings any fixes.
# Safe to re-run: downloads resume, and raw files are deleted only after a successful conversion.
DATA_DRIVE = f"{DRIVE_ROOT}/data/uatd_coco"
LOCAL = "/content/data/uatd_coco"
!python /content/kit/colab/prepare_uatd.py --drive-data "{DATA_DRIVE}" --local-data "{LOCAL}" --pcts {PCTS} --seeds {SEEDS}
os.environ['UATD_DATA_ROOT'] = LOCAL

In [ ]:
#@title 5. Generate the configs inside RSOD
!python /content/kit/tools/gen_rsod_configs.py --rsod-repo /content/RSOD --dataset uatd --coco-train $UATD_DATA_ROOT/annotations/instances_train.json --percents {PCTS} --seeds {SEEDS}

In [ ]:
#@title 6. Train + test (live output; resumes automatically after a disconnect)
# Logic lives in the repo (colab/train.sh). Output is also saved to Drive: work_dirs/console.log
!bash /content/kit/colab/train.sh

In [ ]:
#@title 7. Progress check (run in a second session or after a stop): last log lines per run
import glob, os
for d in sorted(glob.glob(f"{DRIVE_ROOT}/work_dirs/*")):
    logs = sorted(glob.glob(f"{d}/*/*.log"))
    status = 'DONE' if os.path.exists(f"{d}/DONE") else 'running/incomplete'
    print(f"== {os.path.basename(d)}  [{status}]")
    if logs:
        lines = [l for l in open(logs[-1], errors='ignore') if 'Iter(' in l]
        print('   ', lines[-1].strip()[:220] if lines else '(no iterations logged yet)')

In [ ]:
#@title 8. Results vs. the paper (saved to Drive as results.csv)
!python /content/kit/tools/collect_results.py --work-root {DRIVE_ROOT}/work_dirs --out {DRIVE_ROOT}/results.csv